# Лабораторная работа 02. Введение в Pandas: данные об автомобилях

**Курс:** «Анализ данных и искусственный интеллект», 1 курс бакалавриата.

**Задача:** познакомиться с табличными данными и подготовить набор для последующей оценки цен автомобилей на вторичном рынке.

В этой работе мы загрузим данные, выберем нужные строки и колонки, проверим пропуски и повторы, создадим несколько признаков и сохраним результат. Модель предсказания цены пока не строим.

**Нужно знать:** переменные, списки, словари, условия и функции Python.
**После работы вы сможете:** пользоваться `DataFrame` и `Series`, выбирать данные через `loc` и `iloc`, составлять логические фильтры, изменять копию таблицы и сохранять CSV.

**План:** загрузка → знакомство с таблицей → индексация → фильтры → первичная обработка → новые признаки → сохранение.
Разделы 1–8 — основной маршрут. Задания со звёздочкой выполняются самостоятельно.


## 1. Загрузка данных и постановка задачи

Pandas — библиотека Python для работы с таблицами. `DataFrame` представляет двумерную таблицу с именами колонок и индексом строк. `Series` — один столбец значений с индексом.

В наборе данных каждая строка содержит описание автомобиля в предложениях вторичного рынка Молдавии. Главная задача курса — оценить указанную цену по характеристикам автомобиля. Это **регрессия**, поскольку ответ является числом. В качестве дополнительной задачи можно предсказывать тип трансмиссии — это **классификация**.

Сначала исследуем имеющиеся записи. Даты продажи и подтверждённые цены сделок в файле отсутствуют: цена предложения может отличаться от цены сделки.


### Что означают колонки

| Колонка | Содержание | Роль в задаче оценки цены |
|---|---|---|
| `Make` | Марка | Входной признак |
| `Model` | Модель | Входной признак |
| `Year` | Год выпуска | Входной признак |
| `Style` | Тип кузова | Входной признак |
| `Distance` | Пробег, км | Входной признак |
| `Engine_capacity(cm3)` | Объём двигателя, см³ | Входной признак |
| `Fuel_type` | Тип топлива | Входной признак |
| `AutoTransmission` | Автоматическая трансмиссия: True/False | Входной признак |
| `Price(euro)` | Цена, евро | Целевая переменная |

Сохраните `cars.csv` рядом с ноутбуком. В Colab сначала загрузите файл в рабочую папку. Если файла нет, пример загрузится из репозитория курса. Библиотеки при необходимости устанавливаются командой `pip install pandas numpy`.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

data_path = Path('cars.csv')
if data_path.exists():
    raw = pd.read_csv(data_path)
else:
    raw = pd.read_csv(
        'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/'
        'refs/heads/main/Workshops/cars.csv'
    )

# raw остаётся исходной таблицей. Учебные действия выполняем на копии df.
df = raw.copy()
df.head(3)


`pd.read_csv` читает CSV в `DataFrame`. Переменная `raw` хранит исходные данные; `df` — рабочую копию. Такой подход позволяет сравнивать результат обработки с исходным набором.

## 2. Знакомство с таблицей

`head(n)` показывает первые строки, `tail(n)` — последние, `sample(n)` — случайные. `random_state` фиксирует случайный выбор для воспроизводимости. Не нужно выводить весь набор из десятков тысяч строк.


In [ ]:
display(df.head(3))
display(df.tail(3))
display(df.sample(3, random_state=42))


`shape` возвращает пару «число строк, число колонок». `columns` хранит названия колонок, а `index` — метки строк. В исходном файле метки совпадают с порядковыми номерами, но после фильтрации это может измениться.


In [ ]:
print('Размер:', df.shape)
print('Колонки:', df.columns.tolist())
print('Индекс:', df.index)


### Типы данных и непустые значения

`info()` выводит типы колонок и количество непустых значений. `dtypes` показывает типы отдельно. Встречаются целые числа, числа с дробной частью, логические значения и текст. Названия текстовых типов могут отличаться между версиями Pandas (`object` или `str`).

Количество непустых значений меньше числа строк — признак наличия пропусков. Число пропусков нужно вычислить, а не предполагать по нескольким показанным строкам.


In [ ]:
df.info()
display(df.dtypes.to_frame('Тип'))


### Сортировка

`sort_values` возвращает отсортированную таблицу. Исходная `df` не изменяется, пока мы явно не присвоим результат. Сортировку можно сохранять; важно понимать, для чего она нужна. Например, найдём самые дорогие автомобили и самые новые по году выпуска.


In [ ]:
display(df.sort_values('Price(euro)', ascending=False).head(5))
display(df.sort_values('Year', ascending=False).head(5))


### Пропуски, повторы и категории

`isna()` возвращает таблицу True/False: True означает пропуск. `sum()` считает число True по каждой колонке. `duplicated()` отмечает повторные строки после первого совпадения. Без идентификатора объявления одинаковые описания не обязательно означают один и тот же автомобиль — решение об удалении повторов требует допущения.

`value_counts(dropna=False)` показывает частоту категорий, включая пропуски. Для численных значений удобно использовать `describe()`; подробное сравнение групп будет в следующей работе.


In [ ]:
missing_count = df.isna().sum()
display(missing_count.to_frame('Число пропусков'))
print('Повторных строк:', int(df.duplicated().sum()))
display(df['Fuel_type'].value_counts(dropna=False))
display(df[['Year', 'Distance', 'Price(euro)']].describe())


### Задание 1

1. Найдите число строк и число уникальных марок (`nunique()`).
2. Выведите пять самых представленных марок (`value_counts()`).
3. Найдите колонки с пропусками и их долю: число пропусков / число строк.
4. Выведите три самых дорогих автомобиля и назовите параметры, которые требуют проверки.

Пока не удаляйте строки: сначала изучите проблему.


In [ ]:
# Ваш код для задания 1


## 3. Выбор колонок и строк

### Series и DataFrame

`df['Make']` возвращает `Series`, а `df[['Make']]` — `DataFrame` с одной колонкой. Двойные скобки означают передачу списка имён. Выбор нескольких колонок также возвращает `DataFrame`.


In [ ]:
make_series = df['Make']
make_table = df[['Make']]
print(type(make_series), make_series.shape)
print(type(make_table), make_table.shape)
display(df[['Make', 'Model', 'Year', 'Price(euro)']].head(3))


### Индексация loc и iloc

Обе конструкции имеют вид «строки, колонки», но выбирают по-разному:

| Выражение | Что выбирает |
|---|---|
| `df.loc[метка, имя]` | Строку по метке индекса, колонку по имени |
| `df.iloc[номер, номер]` | Строку и колонку по позиции, начиная с нуля |
| `df.loc[0:5, ...]` | Метки от 0 до 5 включительно, если они есть |
| `df.iloc[0:5, ...]` | Позиции от 0 до 4; правая граница исключается |

`loc`/`iloc` не следует объяснять как прямое обращение к ячейкам памяти. Их назначение — выбор данных по меткам или позициям.


In [ ]:
display(df.loc[0:5, ['Make', 'Year', 'Price(euro)']])
display(df.iloc[0:5, 0:3])
display(df.iloc[-1:])


Одна строка может быть `Series` или `DataFrame`: `loc[0]` возвращает строку как `Series`, а `loc[[0]]` сохраняет двумерную таблицу.


In [ ]:
display(df.loc[0, ['Make', 'Model', 'Price(euro)']])
display(df.loc[[0], ['Make', 'Model', 'Price(euro)']])


### Почему метка не всегда совпадает с позицией

В учебном примере зададим строкам метки 10, 20 и 30. `loc[20]` выберет метку 20, а `iloc[1]` — вторую строку. После удаления строк в реальном наборе возникает похожая ситуация.


In [ ]:
index_example = df.head(3).copy()
index_example.index = [10, 20, 30]
display(index_example.loc[20])
display(index_example.iloc[1])


### Задание 2

1. Выберите колонки `Make`, `Model`, `Style`, `Fuel_type`.
2. Выведите последние пять строк только с ценой и годом выпуска.
3. Выберите одну и ту же строку через `loc` и `iloc` в `index_example`.
4. Объясните, почему `index_example.loc[1]` не сработает.


In [ ]:
# Ваш код для задания 2


## 4. Фильтрация по условиям

Сравнение колонки со значением создаёт логическую маску. `df[маска]` оставляет строки с True. Например, выберем автомобили с ручной трансмиссией и отдельно автомобили старше 1970 года выпуска.


In [ ]:
manual_mask = df['AutoTransmission'] == False
display(df.loc[manual_mask, ['Make', 'Model', 'Year', 'AutoTransmission']].head(5))
display(df.loc[df['Year'] < 1970].head(5))


### Несколько условий

Используйте `&` — «и», `|` — «или», `~` — отрицание. Каждое сравнение заключайте в скобки. `and` и `or` предназначены для отдельных логических значений; для масок Pandas нужны `&` и `|`.

Найдём Toyota Land Cruiser Prado 2012 года с автоматической трансмиссией. Количество записей важно: средняя цена одной машины даёт мало информации о сегменте.


In [ ]:
selected = df.loc[
    (df['Make'] == 'Toyota') &
    (df['Model'] == 'Land Cruiser Prado') &
    (df['Year'] == 2012) &
    (df['AutoTransmission'] == True)
].copy()

display(selected[['Make', 'Model', 'Year', 'Price(euro)']])
print('Найдено записей:', len(selected))
if selected.empty:
    print('Для таких условий нет наблюдений.')
else:
    print(f"Средняя цена: {selected['Price(euro)'].mean():.1f} евро")


Тот же отбор можно записать через `query`. Это язык выражений Pandas, а не выполнение SQL-запроса. В основном маршруте достаточно освоить логические маски.


In [ ]:
selected_query = df.query(
    "Make == 'Toyota' and Model == 'Land Cruiser Prado' "
    "and Year == 2012 and AutoTransmission == True"
)
print('Оба способа выбрали те же строки:', selected.index.equals(selected_query.index))


### Проверка необычных сочетаний

Старый год выпуска и автоматическая трансмиссия могут вызвать вопрос, но не доказывают ошибку: возможны исторические особенности, переоборудование или неверное описание. Аналогично крайние цена и пробег требуют проверки.

Правильно отрицать **весь** диапазон: `~((объём >= 200) & (объём <= 5000))`. Отрицание только одной части условия изменяет смысл.


In [ ]:
outside_engine_range = ~df['Engine_capacity(cm3)'].between(200, 5000)
display(df.loc[outside_engine_range,
               ['Make', 'Model', 'Fuel_type', 'Engine_capacity(cm3)']].head(10))

unusual_combination = (df['Year'] < 1950) & (df['AutoTransmission'] == True)
display(df.loc[unusual_combination].head(5))


### Задание 3

1. Выберите Toyota или BMW с ценой не более 10 000 евро.
2. Выберите автомобили 2010–2020 годов выпуска с пробегом 50 000–200 000 км.
3. Найдите записи с неизвестным объёмом двигателя (`isna()`), не смешивая пропуски с известным значением вне диапазона.
4. Напишите функцию `select_cars(table, make, model, year)`, которая возвращает подходящие строки. Проверьте случай, когда совпадений нет.

**Подсказка:** для выбора из нескольких значений используйте `isin(['Toyota', 'BMW'])`.


In [ ]:
# Ваш код для задания 3


## 5. Первичная обработка: один последовательный процесс

Вернёмся к исходной таблице, чтобы результат не зависел от учебных выборок в предыдущих ячейках. Журнал обработки покажет число строк до и после каждого этапа.

### 5.1. Точные повторы

В учебном сценарии считаем полностью совпадающие описания повторами и оставляем первое. Это допущение: без ID объявлений нельзя уверенно отличить повторную выгрузку от разных автомобилей с одинаковыми характеристиками.

`drop_duplicates()` возвращает таблицу без повторов. Индекс пока сохраняем, чтобы можно было найти исходную строку.


In [ ]:
df = raw.copy()
processing_log = []

n_before = len(df)
df = df.drop_duplicates().copy()
processing_log.append({'Этап': 'Удаление точных повторов',
                       'До': n_before, 'После': len(df), 'Убрано': n_before - len(df)})
print('Удалено повторных строк:', n_before - len(df))


### 5.2. Пропуски

Для основной учебной работы используем строки, в которых заполнены все исходные поля. Такое решение упрощает первые упражнения, но уменьшает выборку и может изменить её состав.

Сохраним неполные записи отдельно. В реальной задаче можно оставить часть пропусков или обоснованно заполнить входные признаки. Не заменяйте отсутствующую целевую цену придуманной средней: получится выдуманный ответ для будущей модели.


In [ ]:
incomplete_mask = df.isna().any(axis=1)
incomplete_rows = df.loc[incomplete_mask].copy()
display(incomplete_rows.head(5))

n_before = len(df)
df = df.dropna().copy()
processing_log.append({'Этап': 'Отбор полных записей',
                       'До': n_before, 'После': len(df), 'Убрано': n_before - len(df)})
print('Неполных записей:', len(incomplete_rows))


### 5.3. Область учебного анализа

Сохраним границы отбора из исходной практики, но зададим их в одном месте. Это **учебная область анализа**, а не универсальные признаки ошибочного объявления:

| Условие включения | Как его понимать |
|---|---|
| Год выпуска 1951–2021 | В данном файле максимальный год — 2021; более старые машины оставляем вне учебного сценария |
| Пробег 0–1 000 000 км | Ограничиваем диапазон для первых упражнений |
| Объём двигателя 200–5000 см³ | Фокусируемся на указанном диапазоне; правило может исключить электромобили с нулевым объёмом |
| Цена 101–500 000 евро | Оставляем выбранный диапазон предложений |
| Исключаем пробег менее 1100 км при годе до 2021 | Сохраняем прежнее учебное правило; оно может исключить допустимые малоэксплуатируемые автомобили |

Для других данных границы требуется пересмотреть. Сравним исключённые записи с остающимися; решения нельзя объяснять только редкостью значений.


In [ ]:
MIN_YEAR = 1951
MAX_YEAR = 2021
MIN_ENGINE = 200
MAX_ENGINE = 5000
MIN_PRICE = 101
MAX_PRICE = 500_000
MAX_DISTANCE = 1_000_000
MIN_DISTANCE_OLDER = 1100

in_scope = (
    df['Year'].between(MIN_YEAR, MAX_YEAR) &
    df['Distance'].between(0, MAX_DISTANCE) &
    df['Engine_capacity(cm3)'].between(MIN_ENGINE, MAX_ENGINE) &
    df['Price(euro)'].between(MIN_PRICE, MAX_PRICE) &
    ~((df['Year'] < MAX_YEAR) & (df['Distance'] < MIN_DISTANCE_OLDER))
)

out_of_scope = df.loc[~in_scope].copy()
display(out_of_scope.head(10))
n_before = len(df)
df = df.loc[in_scope].copy()
processing_log.append({'Этап': 'Учебная область анализа',
                       'До': n_before, 'После': len(df), 'Убрано': n_before - len(df)})
print('Осталось строк:', len(df))


### 5.4. Метка исходной строки и новый индекс

При фильтрации индекс сохранял номера исходных строк. Запишем их в `source_row_id`, затем вызовем `reset_index(drop=True)` для последовательной нумерации рабочей таблицы. `source_row_id` — технический номер строки CSV, а не ID реального объявления и не признак для предсказания цены.


In [ ]:
df['source_row_id'] = df.index
df = df.reset_index(drop=True)
display(df[['source_row_id', 'Make', 'Model', 'Year']].head(5))
display(pd.DataFrame(processing_log))


После обработки количество строк уменьшилось. По журналу можно понять, какой этап внёс наибольший вклад. Таблица `raw` позволяет вернуться к исходным значениям.

### Задание 4

1. Рассчитайте долю оставшихся строк от исходного числа.
2. Сравните среднюю и медианную цену до отбора и после него.
3. Выберите три исключённые строки из `out_of_scope` и назовите сработавшее условие. Объясните, мог ли автомобиль быть допустимым.
4. Найдите исходную строку для первого автомобиля в обработанной таблице через `raw.loc[source_row_id]`.

*Дополнительно:* предложите заполнение одного входного признака для неполной записи. Укажите источник оценки и ограничение такого решения.


In [ ]:
# Ваш код для задания 4


## 6. Изменение колонок и создание признаков

### 6.1. Переименование

Сократим названия `Engine_capacity(cm3)` → `Engine`, `Price(euro)` → `Price`. Единицы сохраняются: см³ и евро. `rename` меняет имена, но не значения.


In [ ]:
df = df.rename(columns={'Engine_capacity(cm3)': 'Engine', 'Price(euro)': 'Price'})
print(df.columns.tolist())


### 6.2. Возраст и условный среднегодовой пробег

Создадим `Age` и `km_year` векторными операциями — сразу для всего столбца. Для воспроизведения исходной практики используем **условный опорный год 2022**: максимальный год выпуска в файле + 1.

Это не подтверждённая дата наблюдения и не текущий год. `Age = 2022 − Year`, `km_year = Distance / Age` — условные оценки. Без даты объявления и истории эксплуатации нельзя восстановить реальный пробег за каждый год. Опорный год задаём один раз и не меняем после фильтрации.


In [ ]:
REFERENCE_YEAR = 2022
df['Age'] = REFERENCE_YEAR - df['Year']
df['km_year'] = df['Distance'] / df['Age']
display(df[['Year', 'Age', 'Distance', 'km_year']].head(5))


### 6.3. map: замена по словарю

`map` применяет словарь соответствий к значениям колонки. Создадим понятное текстовое описание трансмиссии. Если в словаре нет ключа, результатом будет пропуск. Исходную `AutoTransmission` пока сохраним.


In [ ]:
transmission_names = {True: 'Automatic', False: 'Manual'}
df['Transmission'] = df['AutoTransmission'].map(transmission_names)
display(df[['AutoTransmission', 'Transmission']].head(5))


### 6.4. apply: применение своей функции

Для упражнения переведём условный годовой пробег в три категории: `Small` — до 10 000 включительно; `Medium` — свыше 10 000 до 30 000 включительно; `Large` — свыше 30 000 км.

`apply` вызывает функцию для каждого значения. У функции есть отдельная ветка для пропуска: неизвестное значение не должно случайно становиться `Large`.


In [ ]:
def km_year_category(value):
    if pd.isna(value):
        return np.nan
    if value <= 10_000:
        return 'Small'
    if value <= 30_000:
        return 'Medium'
    return 'Large'

df['km_year_cat'] = df['km_year'].apply(km_year_category)
display(df[['km_year', 'km_year_cat']].head(5))
display(df['km_year_cat'].value_counts())


### 6.5. Изменение выбранных значений через loc

Для присваивания используйте `table.loc[маска, 'колонка'] = значение`. На отдельной копии покажем, как пометить записи с высоким условным годовым пробегом. Копия не изменяет основную таблицу.

Не используйте цепочку `table[маска]['колонка'] = ...`: такая запись не задаёт надёжное изменение исходной таблицы.


In [ ]:
example = df.head(10).copy()
example['Проверка'] = 'Не отмечено'
example.loc[example['km_year'] > 30_000, 'Проверка'] = 'Высокий условный пробег'
display(example[['source_row_id', 'km_year', 'Проверка']])


### 6.6. Удаление колонок

`drop(columns=...)` удаляет выбранные колонки. Для примера создадим отдельную таблицу без технического номера строки. В основной таблице сохраняем его для проверки происхождения данных.


In [ ]:
example_without_id = df.drop(columns=['source_row_id'])
print('До:', df.shape, 'После:', example_without_id.shape)


### Задание 5

1. Проверьте функцию `km_year_category` на значениях 0, 10 000, 10 001, 30 000, 30 001 и пропуске.
2. Создайте свой категориальный признак по двум или трём диапазонам возраста. Объясните границы.
3. Выделите кандидатов с `km_year > 100_000` в отдельную таблицу; исследуйте возраст и пробег. Не считайте их ошибочными без проверки.
4. Сравните `map`, `apply` и обычную арифметическую операцию по столбцу: для какой задачи подходит каждый способ?


In [ ]:
# Ваш код для задания 5


## 7. Редкие категории: сохраняем исходную информацию

В исходной работе марки с долей не более 1% объединялись в `Rare`. Сохраним этот учебный приём для совместимости с подготовленным примером курса, но сначала скопируем исходную марку в `Make_original`.

Порог 1% — учебное соглашение. Объединение упрощает представление, но скрывает различия редких марок и не гарантирует роста точности. Количество строк не меняется. Для последующего анализа конкретной марки используйте `Make_original`.

В будущем при построении модели список редких категорий нужно определять только по обучающей части; здесь модель ещё не обучается.


In [ ]:
RARE_SHARE = 0.01
make_counts = df['Make'].value_counts()
make_share = make_counts / len(df)
rare_makes = make_share.loc[make_share <= RARE_SHARE].index

df['Make_original'] = df['Make']
df.loc[df['Make'].isin(rare_makes), 'Make'] = 'Rare'

print('Марок объединено:', len(rare_makes))
print('Записей в Rare:', int((df['Make'] == 'Rare').sum()))
display(df[['Make_original', 'Make']].drop_duplicates().head(15))


### Задание 6

1. Найдите число уникальных марок до и после объединения.
2. Посмотрите три исходные марки, попавшие в `Rare`. Какие сведения скрывает общее название?
3. *Дополнительно:* предложите правила поиска редких моделей. Учитывайте, что одинаковое название модели может встретиться у разных марок; не заменяйте все редкие названия без предметного обоснования.


In [ ]:
# Ваш код для задания 6


## 8. Сохранение и передача результата

Сохраним обработанную таблицу в `cars_processed.csv`. Этот файл используется лабораторными 03 и 04. В нём есть базовые характеристики, условный возраст, годовой пробег, категории, исходная марка и номер строки для проверки.

`index=False` запрещает сохранять индекс как лишнюю колонку CSV. Техническая метка `source_row_id` уже сохранена явно.

Также запишем численные колонки в отдельный учебный файл и журнал обработки. Численная таблица **не является готовой матрицей признаков модели**: в ней ещё находится целевая `Price`. Номер строки исключим из неё.


In [ ]:
output_columns = [
    'Make', 'Model', 'Year', 'Style', 'Distance', 'Engine', 'Fuel_type',
    'AutoTransmission', 'Price', 'Age', 'km_year', 'Transmission',
    'km_year_cat', 'Make_original', 'source_row_id'
]
processed = df[output_columns].copy()
processed.to_csv('cars_processed.csv', index=False)
processed.select_dtypes(include='number').drop(columns=['source_row_id']).to_csv(
    'cars_processed_num.csv', index=False
)
pd.DataFrame(processing_log).to_csv('cars_preprocessing_log.csv', index=False)
print('Сохранено:', processed.shape)
print('Колонки:', processed.columns.tolist())


### Контроль результата

Загрузим сохранённый CSV обратно. Проверим размер, пропуски и отсутствие колонки с выгруженным индексом. Типы данных при чтении CSV определяются заново, поэтому для новых файлов их также нужно проверять.


In [ ]:
loaded = pd.read_csv('cars_processed.csv')
print('Совпадает размер:', loaded.shape == processed.shape)
print('Пропусков:', int(loaded.isna().sum().sum()))
print('Лишних колонок индекса:', [c for c in loaded.columns if c.startswith('Unnamed:')])
display(loaded.head(3))


### Входные признаки и целевая переменная

Для будущей регрессии цена — ответ `y`; она не должна одновременно попадать во входные признаки `X`. Пока выберем только исходные характеристики и два вычисленных численных признака.

Не включаем технический номер строки. Текстовая `Transmission` дублирует `AutoTransmission`, а `Make_original` дублирует информацию о марке в другой детализации. Выбор представления и кодирование категорий будут отдельными шагами последующего моделирования.


In [ ]:
feature_columns = [
    'Make', 'Model', 'Year', 'Style', 'Distance', 'Engine',
    'Fuel_type', 'AutoTransmission', 'Age', 'km_year'
]
X = loaded[feature_columns].copy()
y = loaded['Price'].copy()
print('Размер X:', X.shape, 'Размер y:', y.shape)
display(X.head(3))


## 9. Что сдавать

1. Выполненный ноутбук с ответами на задания 1–6.
2. `cars_processed.csv` — подготовленная таблица для следующих работ.
3. `cars_processed_num.csv` — численная часть учебной таблицы.
4. `cars_preprocessing_log.csv` — размеры таблицы на этапах обработки.

В конце ноутбука напишите 5–7 предложений: что представляет одна строка; какой признак является целевым; какие проблемы найдены; сколько строк осталось; какие правила отбора использованы; какие ограничения появились после обработки.

**Оценивание, 10 баллов:** знакомство с данными — 2; выбор строк и колонок — 2; фильтры и обработка — 2; новые признаки и категории — 2; сохранение, журнал и вывод — 2.

Работа подготовила воспроизводимую таблицу и научила выбирать и изменять данные. В лабораторной 03 этот результат используется для группировок и сводного анализа, а в лабораторной 04 — для визуальной проверки и бизнес-отчёта.


In [ ]:
# Ваш итоговый вывод удобно написать в новой Markdown-ячейке.


## Дополнительно: работа с ограниченным сегментом

Напишите функцию, которая для заданных исходной марки, модели и года выпуска возвращает наиболее частый кузов или тип топлива. Используйте `Make_original`, логическую маску, `value_counts()` и проверку пустого результата. Что делать при равных частотах?

Для проверки заполнения пропусков временно скройте несколько **известных входных значений в копии таблицы**. Сравните предложенный ответ с исходным. Не подменяйте корректные редкие значения наиболее частыми автоматически.

## Документация

- [Pandas: первое знакомство](https://pandas.pydata.org/docs/user_guide/10min.html)
- [Выбор данных через loc и iloc](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [Пропуски](https://pandas.pydata.org/docs/user_guide/missing_data.html)
